# Volatility Dynamics 18 — HAR versus Markov conditional-return variance

**Historical exploratory. Not a prospective confirmation or trading signal.**

Frozen research specification: [VD 035](../../discussions/Volatility_Dynamics/035_chatgpt_vd18_har_vs_markov_variance_preregistration.md).

Independent HAR (30m/4h versus 30m/1h/4h/1d) provides predicted terminal-return variance for the *same native five-minute Student-t return process*. This module compares direct HAR variance, HAR blends and HAR-plus-state against VD17 State, VD17 M2-anchor and direct M2. The HAR density calibrators are **horizon overlays**, not one exact joint Markov process.

## 0. Environment and frozen parameters

In [ ]:
import os,sys,time,json,subprocess
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').exists(): ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Start Jupyter from repository tree'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as v14
from data_engineering.modeling import markov_har_variance as v18
DATA=os.getenv('DERIBIT_DATA_ROOT')
assert DATA and Path(DATA).exists(),'Set DERIBIT_DATA_ROOT in repo .env'
HIST=Path(DATA)/'history'/'price_5m'
assert HIST.is_dir(),f'Missing {HIST}'
OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'18'
OUT.mkdir(parents=True,exist_ok=True)
ASSETS=('AVAX','BTC','ETH','HYPE','SOL','TRX','XRP')
assert v18.PARTICLES==256 and v18.BOOT==1000 and v18.PRIMARY==60
START=time.monotonic()
print('VD18 preregistered parameters:',json.dumps(v18.registration(),indent=2))

## 1. Synthetic regression tests — no market history read

In [ ]:
p=subprocess.run([sys.executable,'-m','pytest','-q',
  str(ROOT/'tests'/'test_markov_har_variance.py'),
  str(ROOT/'tests'/'test_markov_conditional_variance.py'),
  str(ROOT/'tests'/'test_markov_return_horizon_calibration.py'),
  str(ROOT/'tests'/'test_student_t_return_markov.py'),
  str(ROOT/'tests'/'test_markov_lift_distributions.py')],
  cwd=str(ROOT),capture_output=True,text=True)
print(p.stdout[-4500:]);print(p.stderr[-1000:]);assert p.returncode==0

## 2. Read five-minute spot history; construct strictly causal HAR lags and censored targets

In [ ]:
raw,excluded=v14.load_returns(HIST,ASSETS)
panel=v14.panelize(raw)
del raw
features=v14.make_features(panel)
frame=v14.make_targets(panel,features,v18.HORIZONS)
frame=v18.add_har_lags(frame)
for h in v18.HORIZONS:
    tr,va,te=v18.complete(frame,h)
    print(f'{h:>3} min: train={len(tr):,} valid={len(va):,} test={len(te):,} days={te.date.nunique()}')
print('Excluded:',excluded,'| elapsed:',round(time.monotonic()-START,1),'seconds')

## 3. Refit VD15–VD17 references; fit HAR train-only forecasts and early-VALID calibrators; score all horizons

In [ ]:
scores,boots,per_asset,har_grid,parameters=v18.run_all(frame,
    horizons=v18.HORIZONS,particles=v18.PARTICLES,B=v18.BOOT,cal_limit=v18.CAL_LIMIT)
assert len(scores)==7*len(v18.MODELS)
assert (scores.groupby('horizon_min').n_origins.nunique()==1).all()
assert (scores.n_days>=30).all()
print('Full historical scoring finished after',round(time.monotonic()-START,1),'seconds')
print('HAR family chosen using late validation:')
display(har_grid[['horizon_min','family','weight','late_valid_nll','direct_late_valid_nll','chosen']])

## 4. Evaluate proper return likelihood, variance QLIKE, and tail calibration

In [ ]:
print('Lower NLL / tail-loss / QLIKE is better; nominal interval miss=.05')
display(scores[['horizon_min','model','nll','return_qlike','tail_logloss','tail_brier','tail_observed','tail_predicted','interval_miss95']].round(6))
primary=boots[(boots.horizon_min==60)&(boots.metric.isin(['nll','return_qlike','tail_logloss']))]
display(primary.round(6))
print('Primary-hour per asset NLL:')
display(per_asset[per_asset.horizon_min==60].pivot(index='asset',columns='model',values='nll').round(6))

## 5. Numerical sensitivity on later VALID only — frozen parameters, 256 versus 512 particles

In [ ]:
sensitivity=v18.numerical_sensitivity_validation(frame,parameters,
    horizons=(5,60,240),particles_low=256,max_per_asset=96)
display(sensitivity.round(6))
print('Largest abs NLL particle-budget change:',sensitivity.high_minus_low.abs().max())

## 6. Decision flags are retrospective; save only small reproducible artifacts

In [ ]:
result=v18.decision(scores,boots,parameters)
assert result['status']=='HISTORICAL_EXPLORATORY'
assert result['prospective_confirmation'] is False
print(json.dumps(result,indent=2))
scores.to_csv(OUT/'18_scores.csv',index=False,float_format='%.10g')
boots.to_csv(OUT/'18_bootstrap.csv',index=False,float_format='%.10g')
per_asset.to_csv(OUT/'18_per_asset.csv',index=False,float_format='%.10g')
har_grid.to_csv(OUT/'18_har_family_selection.csv',index=False,float_format='%.10g')
sensitivity.to_csv(OUT/'18_particle_sensitivity.csv',index=False,float_format='%.10g')
(OUT/'18_decision.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
manifest={'module':'Volatility_Dynamics_18','registration':v18.registration(),
 'assets':sorted(frame.underlying.unique().tolist()),'excluded_assets':excluded,
 'sample_first':str(frame.state_time.min()),'sample_last':str(frame.state_time.max()),
 'days_by_split':{str(k):int(v) for k,v in frame.groupby('split').date.nunique().items()},
 'parameters':parameters,'decision':result,
 'note':'Previously inspected historical TEST: model selection exploratory. HAR overlay is not a new Markov return process.'}
(OUT/'18_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved 7 compact outputs to',OUT)
print('No origin-level forecasts, particle grids, ticks or market data were saved.')

## 7. Interpretation guide

- **HAR_DIRECT over VD17 STATE and M2:** independent HAR variance information is useful without borrowing M2 variance; check QLIKE and both tail scores.
- **HAR_BLEND over HAR_DIRECT:** unrestricted HAR variance was too aggressive, or its calibration sample did not match the test distribution.
- **HAR_STATE over HAR_BLEND:** continuous Markov-volatility state adds information beyond HAR and retains a reason to develop a HAR-augmented stochastic process.
- **M2 still wins:** do not assume a harder generative problem will beat direct conditional-density fitting. Report no trading or market-model confirmation from the already inspected sample.

The native five-minute Student-t return law stays exactly as VD15; *no latent two-state chain is fitted in this module*. A discrete regime follow-up requires independent improvement over the best continuous volatility engine.